# Results Visualization

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import defaultdict

# Plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Setup
%load_ext autoreload
%autoreload 2

# Data paths
RUN_ID = "20260315_gpt_4_1_beginner_test2"
RUN_PATH = Path("outputs") / RUN_ID
DATA_PATH = Path("data")

print(f"Loading data from: {RUN_PATH}\n")

## Difficulty vs turns

In [ ]:
from utils.data import load_leetcodedataset_data, load_or_create_shuffled_data
import matplotlib.pyplot as plt

# df_train, df_test = load_leetcodedataset_data(DATA_PATH)
df_train, df_test = load_or_create_shuffled_data(DATA_PATH)
df_sample = df_train.head(100).copy()
df_sample.head()

In [ ]:
# Configuration: group size (number of problems per group)
group_size = 10  # Change this to adjust grouping

# Prepare data for stacked plot
df_sample['group'] = (df_sample.index // group_size).astype(int)

# Count difficulty levels per group
difficulty_counts = df_sample.groupby(['group', 'difficulty']).size().unstack(fill_value=0)

# Create group labels
group_labels = [f"Problems {i*group_size}-{(i+1)*group_size-1}" for i in range(len(difficulty_counts))]

# Define colors for difficulty levels
colors = {
    'Easy': 'rgba(76, 175, 80, 0.8)',      # Green
    'Medium': 'rgba(255, 193, 7, 0.8)',    # Amber
    'Hard': 'rgba(244, 67, 54, 0.8)'       # Red
}

# Create stacked bar chart
fig = go.Figure()

for difficulty in ['Easy', 'Medium', 'Hard']:
    if difficulty in difficulty_counts.columns:
        fig.add_trace(go.Bar(
            x=group_labels,
            y=difficulty_counts[difficulty],
            name=difficulty.capitalize(),
            marker=dict(color=colors.get(difficulty, 'gray')),
            hovertemplate='<b>%{x}</b><br>' + difficulty.capitalize() + ': %{y}<extra></extra>'
        ))

fig.update_layout(
    title=f"Problem Difficulty Distribution Across Training (Grouped by {group_size} Problems)",
    xaxis_title="Training Progress",
    yaxis_title="Count",
    barmode='stack',
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    hovermode='x unified'
)

fig.show()

print(f"Difficulty Distribution (group size={group_size}):")
print(difficulty_counts)

## Load Checkpoints Data

In [ ]:
# Load memory (RL state transitions)
memory_path = RUN_PATH / "checkpoints" / "memory.csv"
df_memory = pd.read_csv(memory_path)
print(f"Loaded {len(df_memory)} transitions from memory.csv")
print(f"Columns: {df_memory.columns.tolist()}")
print(f"\nShape: {df_memory.shape}")
print(f"\nFirst few rows:")
df_memory.head()

In [ ]:
# Load interaction files to understand structure
interactions_dir = RUN_PATH / "interactions"
interaction_files = sorted(interactions_dir.glob("problem_*.json"))
print(f"Found {len(interaction_files)} interaction files")

# Load one example to explore structure
with open(interaction_files[0]) as f:
    sample_interaction = json.load(f)

print(f"\nSample interaction (problem 0) has {len(sample_interaction)} entries")
print("Structure:")
for i, entry in enumerate(sample_interaction[:3]):
    print(f"  Entry {i}: {list(entry.keys())}")

# Graph 3: Tutor and Student Abstraction Level Distributions

In [ ]:
def plot_abstraction_level_distribution(df, level_col, title, color_rgb, entity_name):
    """
    Create a bar chart for abstraction level distribution.
    
    Parameters:
    - df: DataFrame
    - level_col: column name (e.g., 'tutor_level', 'student_level')
    - title: chart title
    - color_rgb: color string (e.g., 'rgba(255, 182, 193, 0.8)')
    - entity_name: name of entity (e.g., 'Tutor Level', 'Student Level')
    """
    level_counts = df[level_col].value_counts().sort_index()
    
    fig = go.Figure(data=[
        go.Bar(
            x=level_counts.index.astype(str),
            y=level_counts.values,
            marker=dict(color=color_rgb, line=dict(color='darkred', width=2)),
            text=level_counts.values,
            textposition='auto',
            hovertemplate='<b>Level %{x}</b><br>Count: %{y}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title=title,
        xaxis_title=f"{entity_name} (1=Concrete, 4=Abstract)",
        yaxis_title="Frequency",
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=700,
        font=dict(size=12),
        showlegend=False
    )
    
    fig.show()
    
    print(f"\n{entity_name} Statistics:")
    print(f"Mean: {df[level_col].mean():.2f}")
    print(f"Median: {df[level_col].median():.2f}")
    print(f"Std Dev: {df[level_col].std():.2f}")
    print(f"\nCounts:\n{level_counts}")

# Plot tutor level distribution
plot_abstraction_level_distribution(
    df_memory,
    'tutor_level',
    'Judged Tutor Abstraction Levels',
    'rgba(255, 182, 193, 0.8)',
    'Tutor Level'
)

In [ ]:
# Plot student level distribution
plot_abstraction_level_distribution(
    df_memory,
    'student_level',
    'Judged Student Abstraction Levels',
    'rgba(135, 206, 250, 0.8)',
    'Student Level'
)

In [ ]:
# Create cross-tabulation (confusion matrix) of student_level vs tutor_level from automated judgments
print(f"Creating confusion matrix from {len(df_memory)} automated transitions\n")

# Create cross-tabulation
confusion_matrix = pd.crosstab(df_memory['student_level'], df_memory['tutor_level'])
print("Student Level vs Tutor Level Cross-Tabulation (Automated Judgments):")
print(confusion_matrix)
print(f"\nShape: {confusion_matrix.shape}")

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=confusion_matrix.values,
    x=[f"Tutor Level {i}" for i in confusion_matrix.columns],
    y=[f"Student Level {i}" for i in confusion_matrix.index],
    text=confusion_matrix.values,
    texttemplate='%{text}',
    textfont={"size": 14},
    colorscale='Blues',
    colorbar=dict(title="Count"),
    hovertemplate='<b>%{y}</b><br><b>%{x}</b><br>Count: %{z}<extra></extra>'
))

fig.update_layout(
    title="Judge Alignment: Student Level vs Tutor Level (Automated Judgments)",
    xaxis_title="Tutor Abstraction Level (1=Concrete, 4=Abstract)",
    yaxis_title="Student Abstraction Level (1=Concrete, 4=Abstract)",
    height=600,
    width=750,
    font=dict(size=12),
)

fig.show()

# Print statistics
print("\n" + "="*60)
print("JUDGE ALIGNMENT ANALYSIS")
print("="*60)
print(f"\nMatching levels (where student_level == tutor_level):")
matching = sum(df_memory['student_level'] == df_memory['tutor_level'])
pct = (matching / len(df_memory)) * 100
print(f"  Count: {matching} ({pct:.1f}%)")

print(f"\nStudent ahead of tutor (student_level > tutor_level):")
ahead = sum(df_memory['student_level'] > df_memory['tutor_level'])
pct = (ahead / len(df_memory)) * 100
print(f"  Count: {ahead} ({pct:.1f}%)")

print(f"\nTutor ahead of student (tutor_level > student_level):")
tutor_ahead = sum(df_memory['tutor_level'] > df_memory['student_level'])
pct = (tutor_ahead / len(df_memory)) * 100
print(f"  Count: {tutor_ahead} ({pct:.1f}%)")

print(f"\nLevel difference (abs):")
level_diff = abs(df_memory['student_level'] - df_memory['tutor_level'])
print(f"  Mean: {level_diff.mean():.2f}")
print(f"  Median: {level_diff.median():.2f}")
print(f"  Max: {level_diff.max():.0f}")


# Graph 10: Pedagogical Move Distribution

In [ ]:
# Get pedagogical move distribution
action_counts = df_memory['action'].value_counts()
action_order = ["SOCRATIC_PROBE", "CONCEPTUAL_HINT", "STRUCTURAL_SCAFFOLD"]
action_counts = action_counts.reindex([a for a in action_order if a in action_counts.index])

# Color mapping for actions
action_colors = {
    "SOCRATIC_PROBE": "rgba(100, 149, 237, 0.8)",        # Cornflower blue
    "CONCEPTUAL_HINT": "rgba(144, 238, 144, 0.8)",       # Light green
    "STRUCTURAL_SCAFFOLD": "rgba(255, 165, 0, 0.8)",     # Orange
}

fig = go.Figure(data=[
    go.Bar(
        x=action_counts.index,
        y=action_counts.values,
        marker=dict(
            color=[action_colors.get(action, 'gray') for action in action_counts.index],
            line=dict(color='black', width=1.5)
        ),
        text=action_counts.values,
        textposition='auto',
        hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
    )
])

fig.update_layout(
    title="Distribution of Pedagogical Actions (Training Cycle)",
    xaxis_title="Pedagogical Move",
    yaxis_title="Frequency",
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    height=500,
    width=800,
    font=dict(size=12),
    showlegend=False,
    xaxis=dict(tickangle=-15)
)

fig.show()

print(f"\nPedagogical Move Statistics:")
print(f"Total actions: {action_counts.sum()}")
print(f"\nCounts:")
for action, count in action_counts.items():
    pct = (count / action_counts.sum()) * 100
    print(f"  {action}: {count} ({pct:.1f}%)")

# Graph 10.2: Action with Highest Predicted Reward

In [ ]:
# Extract best actions from predicted rewards
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
df_with_preds = df_memory[pred_columns].dropna()

if len(df_with_preds) > 0:
    # Find which action had the highest predicted reward for each row
    best_actions = df_with_preds.idxmax(axis=1).str.replace('pred_reward_', '')
    best_action_counts = best_actions.value_counts()
    best_action_counts = best_action_counts.reindex([a for a in action_order if a in best_action_counts.index])
    
    # Create visualization
    fig = go.Figure(data=[
        go.Bar(
            x=best_action_counts.index,
            y=best_action_counts.values,
            marker=dict(
                color=[action_colors.get(action, 'gray') for action in best_action_counts.index],
                line=dict(color='black', width=1.5)
            ),
            text=best_action_counts.values,
            textposition='auto',
            hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title="Distribution of Optimal Actions (Exploitation Cycle)",
        xaxis_title="Pedagogical Move",
        yaxis_title="Frequency",
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=800,
        font=dict(size=12),
        showlegend=False,
        xaxis=dict(tickangle=-15)
    )
    
    fig.show()
    
    # Print statistics
    print(f"\nHighest Predicted Reward Action Statistics:")
    print(f"Rows with predictions: {len(df_with_preds)}")
    print(f"\nCounts:")
    for action, count in best_action_counts.items():
        pct = (count / best_action_counts.sum()) * 100
        print(f"  {action}: {count} ({pct:.1f}%)")
else:
    print("No rows with predicted rewards found")

# Graph 6: Highest Predicted Action Distribution Over Training Time

In [ ]:
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
group_size = 10  # Group every N problems

# Extract best actions from predictions
df_with_preds = df_memory[pred_columns].dropna()

if len(df_with_preds) > 0:
    best_actions = df_with_preds.idxmax(axis=1).str.replace('pred_reward_', '')
    problem_nums = df_memory.loc[best_actions.index, 'problem'].values
    
    # Group by training progress
    df_grouped = pd.DataFrame({
        'best_action': best_actions.values,
        'problem': problem_nums
    })
    df_grouped['group'] = (df_grouped['problem'] // group_size).astype(int)
    
    group_action_counts = df_grouped.groupby(['group', 'best_action']).size().unstack(fill_value=0)
    group_action_counts = group_action_counts.reindex([a for a in action_order if a in group_action_counts.columns], axis=1, fill_value=0)
    
    # Create visualization
    group_labels = [f"Problems {i*group_size}-{(i+1)*group_size-1}" for i in group_action_counts.index]
    
    fig = go.Figure()
    for action in group_action_counts.columns:
        fig.add_trace(go.Scatter(
            x=group_labels,
            y=group_action_counts[action],
            mode='lines',
            name=action,
            line=dict(width=0),
            fillcolor=action_colors.get(action, 'gray'),
            stackgroup='one',
            hovertemplate='<b>%{fullData.name}</b><br>%{x}<br>Count: %{y}<extra></extra>'
        ))
    
    fig.update_layout(
        title=f"Distribution of Optimal Actions Over Training (Grouped by {group_size} Problems)",
        xaxis_title="Training Progress",
        yaxis_title="Frequency",
        hovermode='x unified',
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=900,
        font=dict(size=11),
    )
    
    fig.show()
    
    # Print summary
    print(f"\nHighest Predicted Action Over Training Time (group size={group_size}):")
    print(f"Total groups: {len(group_action_counts)}")
    print(f"\n{group_action_counts}")
else:
    print("No rows with predicted rewards found")

# Graph 1: Conversation End Reasons (Pie Chart)

In [ ]:
import json
from collections import Counter

# Extract stop reasons from all interaction files
stop_reasons = []
stop_explanations = Counter()

for interaction_file in interaction_files:
    with open(interaction_file, 'r') as f:
        interactions = json.load(f)
    
    # Find the stop entry (should be last or near end)
    for entry in interactions:
        if isinstance(entry, dict) and "stop" in entry:
            stop_info = entry["stop"]
            
            # Determine the reason
            if "error" in stop_info:
                reason = "Error"
            elif "problem_finished" in stop_info:
                if stop_info["problem_finished"]:
                    reason = "Problem Solved"
                else:
                    explanation = stop_info.get("explanation", "Unknown")
                    reason = explanation
            else:
                reason = "Unknown"
            
            stop_reasons.append(reason)
            stop_explanations[reason] += 1

# Aggregate similar reasons
reason_counts = Counter(stop_reasons)
print("Stop Reasons Distribution:")
for reason, count in reason_counts.most_common():
    pct = (count / len(stop_reasons)) * 100
    print(f"  {reason}: {count} ({pct:.1f}%)")

# Create pie chart
colors = {
    "Problem Solved": "rgba(76, 175, 80, 0.8)",           # Green
    "Max number of iterations": "rgba(255, 193, 7, 0.8)", # Amber
    "Leakage detected": "rgba(244, 67, 54, 0.8)",         # Red
    "Student changed problem": "rgba(233, 30, 99, 0.8)",  # Pink
    "Error": "rgba(156, 39, 176, 0.8)",                   # Purple
}

fig = go.Figure(data=[
    go.Pie(
        labels=list(reason_counts.keys()),
        values=list(reason_counts.values()),
        marker=dict(
            colors=[colors.get(reason, 'rgba(158, 158, 158, 0.8)') for reason in reason_counts.keys()],
            line=dict(color='white', width=2)
        ),
        textposition='inside',
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
    )
])

fig.update_layout(
    title="How Conversations Ended (100 Problems)",
    height=600,
    width=700,
    font=dict(size=12),
    showlegend=True
)

fig.show()

print(f"\nTotal problems analyzed: {len(stop_reasons)}")

# Conversation End Reasons by Problem Difficulty

In [ ]:
# Extract stop reasons grouped by problem difficulty
stop_data = []

for problem_idx, interaction_file in enumerate(interaction_files):
    with open(interaction_file, 'r') as f:
        interactions = json.load(f)
    
    # Get difficulty for this problem
    if problem_idx < len(df_train):
        difficulty = df_train.iloc[problem_idx]['difficulty']
    else:
        difficulty = 'Unknown'
    
    # Find the stop entry
    for entry in interactions:
        if isinstance(entry, dict) and "stop" in entry:
            stop_info = entry["stop"]
            
            # Determine the reason
            if "error" in stop_info:
                reason = "Error"
            elif "problem_finished" in stop_info:
                if stop_info["problem_finished"]:
                    reason = "Problem Solved"
                else:
                    explanation = stop_info.get("explanation", "Unknown")
                    reason = explanation
            else:
                reason = "Unknown"
            
            stop_data.append({
                'problem': problem_idx,
                'difficulty': difficulty,
                'stop_reason': reason
            })
            break

# Create DataFrame
df_stops = pd.DataFrame(stop_data)

print(f"Stop data by difficulty:")
print(df_stops['difficulty'].value_counts())
print(f"\nCrosstab of difficulty vs stop reason:")
stop_by_difficulty = pd.crosstab(df_stops['difficulty'], df_stops['stop_reason'])
print(stop_by_difficulty)

# Create stacked bar chart
difficulty_order = ['Easy', 'Medium', 'Hard']
stop_by_difficulty = stop_by_difficulty.reindex([d for d in difficulty_order if d in stop_by_difficulty.index])

# Normalize to proportions (0-100%)
stop_by_difficulty_pct = stop_by_difficulty.div(stop_by_difficulty.sum(axis=1), axis=0) * 100

# Color mapping for stop reasons
stop_colors = {
    "Problem Solved": "rgba(76, 175, 80, 0.8)",
    "Max number of iterations": "rgba(255, 193, 7, 0.8)",
    "Leakage detected": "rgba(244, 67, 54, 0.8)",
    "Student changed problem": "rgba(233, 30, 99, 0.8)",
    "Error": "rgba(156, 39, 176, 0.8)",
}

fig = go.Figure()

for reason in stop_by_difficulty_pct.columns:
    # Prepare text labels showing both count and percentage
    text_labels = []
    for difficulty in stop_by_difficulty_pct.index:
        count = stop_by_difficulty.loc[difficulty, reason]
        pct = stop_by_difficulty_pct.loc[difficulty, reason]
        text_labels.append(f"{int(count)}<br>{pct:.1f}%")
    
    fig.add_trace(go.Bar(
        x=stop_by_difficulty_pct.index,
        y=stop_by_difficulty[reason],
        name=reason,
        marker=dict(color=stop_colors.get(reason, 'rgba(158, 158, 158, 0.8)')),
        text=text_labels,
        textposition='inside',
        hovertemplate='<b>%{x}</b><br>' + reason + ': %{y} (%{customdata:.1f}%)<extra></extra>',
        customdata=stop_by_difficulty_pct[reason].values
    ))

fig.update_layout(
    title="Conversation End Reasons by Problem Difficulty",
    xaxis_title="Problem Difficulty",
    yaxis_title="Count",
    barmode='stack',
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    height=500,
    width=900,
    font=dict(size=11),
    hovermode='x unified'
)

fig.show()

# Print breakdown by difficulty
print(f"\n{'='*60}")
print(f"Conversation End Reasons by Difficulty:")
print(f"{'='*60}")
for difficulty in difficulty_order:
    if difficulty in stop_by_difficulty.index:
        total = stop_by_difficulty.loc[difficulty].sum()
        print(f"\n{difficulty} Problems (total: {total}):")
        for reason in stop_by_difficulty.columns:
            count = stop_by_difficulty.loc[difficulty, reason]
            pct = (count / total * 100) if total > 0 else 0
            print(f"  {reason:25s}: {count:3d} ({pct:5.1f}%)")

# Reward vs Conversation Turns

In [ ]:
# Cumulative Reward vs Turns
# Load memory fresh and compute cumulative metrics per turn
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Group by turn index (i) and calculate reward statistics
reward_by_turn = df.groupby('i')['reward'].agg(['mean', 'median', 'std', 'count']).reset_index()

print("Cumulative Reward by Turn Index:")
print(reward_by_turn)
print(f"\nTotal turns: {len(reward_by_turn)}")

# Create plot with mean and median lines
fig = go.Figure()

# Mean line
fig.add_trace(go.Scatter(
    x=reward_by_turn['i'],
    y=reward_by_turn['mean'],
    mode='lines+markers',
    name='Mean Reward',
    line=dict(color='rgba(255, 152, 0, 1)', width=3),
    marker=dict(size=8),
    hovertemplate='Turn %{x}: Mean Reward = %{y:.3f}<extra></extra>'
))

# Fill between mean ± std
fig.add_trace(go.Scatter(
    x=reward_by_turn['i'].tolist() + reward_by_turn['i'].tolist()[::-1],
    y=(reward_by_turn['mean'] + reward_by_turn['std']).tolist() + (reward_by_turn['mean'] - reward_by_turn['std']).tolist()[::-1],
    fill='toself',
    fillcolor='rgba(255, 152, 0, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='±1 Std Dev',
    hoverinfo='skip'
))

fig.update_layout(
    title="RL Reward Signal vs Conversation Turn",
    xaxis_title="Turn Index (0 = first tutor response)",
    yaxis_title="Reward (student_success + pedagogical_quality)",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    hovermode='x unified',
)

fig.show()

print(f"\nInsights:")
print(f"  Turn 0 mean reward: {reward_by_turn.iloc[0]['mean']:.3f}")
print(f"  Best turn: {reward_by_turn.loc[reward_by_turn['mean'].idxmax(), 'i']:.0f} with {reward_by_turn['mean'].max():.3f}")
print(f"  Worst turn: {reward_by_turn.loc[reward_by_turn['mean'].idxmin(), 'i']:.0f} with {reward_by_turn['mean'].min():.3f}")
print(f"  Overall trend: {'Improving' if reward_by_turn.iloc[-1]['mean'] > reward_by_turn.iloc[0]['mean'] else 'Declining'}")


# Performance Metrics vs Conversation Turns

In [ ]:
# Student Success vs Turns
# Load memory fresh and group by turn index
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Group by turn index (i) and calculate statistics
success_by_turn = df.groupby('i')['student_success'].agg(['mean', 'median', 'std', 'count']).reset_index()

print("Student Success by Turn Index:")
print(success_by_turn)
print(f"\nTotal turns: {len(success_by_turn)}")

# Create plot with mean and median lines
fig = go.Figure()

# Mean line
fig.add_trace(go.Scatter(
    x=success_by_turn['i'],
    y=success_by_turn['mean'],
    mode='lines+markers',
    name='Mean Success',
    line=dict(color='rgba(76, 175, 80, 1)', width=3),
    marker=dict(size=8),
    hovertemplate='Turn %{x}: Mean Success = %{y:.2%}<extra></extra>'
))

# Fill between mean ± std
fig.add_trace(go.Scatter(
    x=success_by_turn['i'].tolist() + success_by_turn['i'].tolist()[::-1],
    y=(success_by_turn['mean'] + success_by_turn['std']).tolist() + (success_by_turn['mean'] - success_by_turn['std']).tolist()[::-1],
    fill='toself',
    fillcolor='rgba(76, 175, 80, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='±1 Std Dev',
    hoverinfo='skip'
))

fig.update_layout(
    title="Student Code Success vs Conversation Turn",
    xaxis_title="Turn Index (0 = first tutor response)",
    yaxis_title="Student Success Rate (% tests passed)",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    hovermode='x unified',
)

fig.update_yaxes(tickformat='.0%')
fig.show()

print(f"\nInsights:")
print(f"  Turn 0 mean success: {success_by_turn.iloc[0]['mean']:.1%}")
print(f"  Best turn: {success_by_turn.loc[success_by_turn['mean'].idxmax(), 'i']:.0f} with {success_by_turn['mean'].max():.1%}")
print(f"  Worst turn: {success_by_turn.loc[success_by_turn['mean'].idxmin(), 'i']:.0f} with {success_by_turn['mean'].min():.1%}")


In [ ]:
# Model Training Progress Across 100 Conversations
# Load memory fresh
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 10  # Number of problems to group (configurable)

# Calculate mean accuracy per problem (0-99)
accuracy_by_problem = df.groupby('problem')['student_success'].agg(['mean', 'std', 'count']).reset_index()
accuracy_by_problem.columns = ['problem', 'accuracy_mean', 'accuracy_std', 'count']

# Group problems into chunks and calculate mean accuracy per group
accuracy_by_problem['group'] = (accuracy_by_problem['problem'] // group_size_problems).astype(int)
grouped_accuracy = accuracy_by_problem.groupby('group').agg({
    'accuracy_mean': ['mean', 'std'],
    'count': 'sum'
}).reset_index()

# Flatten column names
grouped_accuracy.columns = ['group', 'mean', 'std', 'total_transitions']

# Create group labels
grouped_accuracy['label'] = grouped_accuracy['group'].apply(
    lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
)

print(f"Model Training Progress Across 100 Conversations (group size = {group_size_problems} problems):")
print(grouped_accuracy[['label', 'mean', 'std', 'total_transitions']])

# Create bar chart with error bars
fig = go.Figure(data=[
    go.Bar(
        x=grouped_accuracy['label'],
        y=grouped_accuracy['mean'],
        error_y=dict(
            type='data',
            array=grouped_accuracy['std'],
            visible=True
        ),
        marker=dict(
            color='rgba(66, 133, 244, 0.8)',
            line=dict(color='rgba(25, 103, 210, 1)', width=2)
        ),
        text=[f"{v:.1%}" for v in grouped_accuracy['mean']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Mean Accuracy: %{y:.2%}<br>Std Dev: %{error_y.array:.2%}<br>Transitions: %{customdata}<extra></extra>',
        customdata=grouped_accuracy['total_transitions']
    )
])

fig.update_layout(
    title=f"Model Training Progress: Student Accuracy Across 100 Conversations (Grouped by {group_size_problems} Problems)",
    xaxis_title="Training Conversation Progress",
    yaxis_title="Mean Student Success Rate",
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    showlegend=False
)

fig.update_yaxes(tickformat='.0%', range=[0, 1.0])
fig.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"Training Progress Summary (group size = {group_size_problems} problems):")
print(f"{'='*60}")
print(f"Total groups: {len(grouped_accuracy)}")
print(f"First group (early training) mean accuracy: {grouped_accuracy.iloc[0]['mean']:.1%}")
print(f"Last group (late training) mean accuracy: {grouped_accuracy.iloc[-1]['mean']:.1%}")
improvement = grouped_accuracy.iloc[-1]['mean'] - grouped_accuracy.iloc[0]['mean']
print(f"Overall improvement: {improvement:+.1%}")
best_idx = grouped_accuracy['mean'].idxmax()
print(f"Best group: {grouped_accuracy.loc[best_idx, 'label']} ({grouped_accuracy.loc[best_idx, 'mean']:.1%})")
worst_idx = grouped_accuracy['mean'].idxmin()
print(f"Worst group: {grouped_accuracy.loc[worst_idx, 'label']} ({grouped_accuracy.loc[worst_idx, 'mean']:.1%})")


# Problem Solved Rate vs Training Turns

In [ ]:
# Calculate problem solved rate by training progress
# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 10  # Change this to adjust grouping

# Create a dataframe with problem index and whether it was solved
df_stops_indexed = df_stops.copy()
df_stops_indexed['is_solved'] = (df_stops_indexed['stop_reason'] == 'Problem Solved').astype(int)

# Group by chunks of problems
df_stops_indexed['group'] = (df_stops_indexed['problem'] // group_size_problems).astype(int)

# Calculate solved rate per group
solved_rate_by_group = df_stops_indexed.groupby('group').agg({
    'is_solved': ['sum', 'count', 'mean']
}).reset_index()

# Flatten column names
solved_rate_by_group.columns = ['group', 'solved_count', 'total_count', 'solved_rate']
solved_rate_by_group['solved_rate'] = solved_rate_by_group['solved_rate'] * 100

# Create group labels
solved_rate_by_group['label'] = solved_rate_by_group['group'].apply(
    lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
)

print(f"Problem Solved Rate by Training Progress (group size = {group_size_problems}):")
print(solved_rate_by_group[['label', 'solved_count', 'total_count', 'solved_rate']])
print()

# Create visualization
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=solved_rate_by_group['label'],
    y=solved_rate_by_group['solved_rate'],
    mode='lines+markers',
    name='Success Rate',
    line=dict(color='rgba(76, 175, 80, 1)', width=3),
    marker=dict(size=10),
    fill='tozeroy',
    fillcolor='rgba(76, 175, 80, 0.2)',
    text=[f"{rate:.1f}%<br>{int(solved)}/{int(total)}" for rate, solved, total in 
          zip(solved_rate_by_group['solved_rate'], solved_rate_by_group['solved_count'], solved_rate_by_group['total_count'])],
    textposition='top center',
    hovertemplate='<b>%{x}</b><br>Success Rate: %{y:.1f}%<br>Solved: %{customdata[0]}/{%{customdata[1]}}<extra></extra>',
    customdata=solved_rate_by_group[['solved_count', 'total_count']].values
))

fig.update_layout(
    title=f"Problem Solved Rate Over Training Progress (Grouped by {group_size_problems} Problems)",
    xaxis_title="Training Progress",
    yaxis_title="Success Rate (%)",
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=11),
    yaxis=dict(range=[0, 105]),
    hovermode='x unified'
)

fig.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"Problem Solved Rate by Training Progress (group size = {group_size_problems}):")
print(f"{'='*60}")
for _, row in solved_rate_by_group.iterrows():
    print(f"  {row['label']:25s}: {row['solved_rate']:5.1f}% ({int(row['solved_count'])}/{int(row['total_count'])} problems)")

In [ ]:
# Model Training Progress: Cumulative Reward Across 100 Conversations
# Load memory fresh
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 10  # Number of problems to group (configurable)

# Calculate mean reward per problem (0-99)
reward_by_problem = df.groupby('problem')['reward'].agg(['mean', 'std', 'count']).reset_index()
reward_by_problem.columns = ['problem', 'reward_mean', 'reward_std', 'count']

# Group problems into chunks and calculate mean reward per group
reward_by_problem['group'] = (reward_by_problem['problem'] // group_size_problems).astype(int)
grouped_reward = reward_by_problem.groupby('group').agg({
    'reward_mean': ['mean', 'std'],
    'count': 'sum'
}).reset_index()

# Flatten column names
grouped_reward.columns = ['group', 'mean', 'std', 'total_transitions']

# Create group labels
grouped_reward['label'] = grouped_reward['group'].apply(
    lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
)

print(f"Model Training Progress - Cumulative Reward (group size = {group_size_problems} problems):")
print(grouped_reward[['label', 'mean', 'std', 'total_transitions']])

# Create bar chart with error bars
fig = go.Figure(data=[
    go.Bar(
        x=grouped_reward['label'],
        y=grouped_reward['mean'],
        error_y=dict(
            type='data',
            array=grouped_reward['std'],
            visible=True
        ),
        marker=dict(
            color='rgba(255, 152, 0, 0.8)',
            line=dict(color='rgba(230, 124, 0, 1)', width=2)
        ),
        text=[f"{v:.3f}" for v in grouped_reward['mean']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Mean Reward: %{y:.4f}<br>Std Dev: %{error_y.array:.4f}<br>Transitions: %{customdata}<extra></extra>',
        customdata=grouped_reward['total_transitions']
    )
])

fig.update_layout(
    title=f"Model Training Progress: Cumulative Reward Across 100 Conversations (Grouped by {group_size_problems} Problems)",
    xaxis_title="Training Conversation Progress",
    yaxis_title="Mean Reward Signal",
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    showlegend=False
)

fig.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"Training Progress Summary - Reward (group size = {group_size_problems} problems):")
print(f"{'='*60}")
print(f"Total groups: {len(grouped_reward)}")
print(f"First group (early training) mean reward: {grouped_reward.iloc[0]['mean']:.4f}")
print(f"Last group (late training) mean reward: {grouped_reward.iloc[-1]['mean']:.4f}")
improvement = grouped_reward.iloc[-1]['mean'] - grouped_reward.iloc[0]['mean']
print(f"Overall change: {improvement:+.4f}")
best_idx = grouped_reward['mean'].idxmax()
print(f"Best group: {grouped_reward.loc[best_idx, 'label']} ({grouped_reward.loc[best_idx, 'mean']:.4f})")
worst_idx = grouped_reward['mean'].idxmin()
print(f"Worst group: {grouped_reward.loc[worst_idx, 'label']} ({grouped_reward.loc[worst_idx, 'mean']:.4f})")

In [ ]:
# Model Training Progress: Reward Model Mean Absolute Error (MAE) Across 100 Conversations
# Load memory fresh
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 5  # Number of problems to group (configurable)

# Compute MAE for each transition
# MAE = |actual_reward - predicted_reward|
# For predicted reward, use the best (max) predicted reward among the three actions
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
df_with_preds = df[pred_columns + ['reward', 'problem']].dropna()

if len(df_with_preds) > 0:
    # Get best predicted reward for each row
    best_pred_reward = df_with_preds[pred_columns].max(axis=1)
    
    # Compute MAE
    df_with_preds['mae'] = abs(df_with_preds['reward'] - best_pred_reward)
    
    # Calculate mean MAE per problem (0-99)
    mae_by_problem = df_with_preds.groupby('problem')['mae'].agg(['mean', 'std', 'count']).reset_index()
    mae_by_problem.columns = ['problem', 'mae_mean', 'mae_std', 'count']
    
    # Group problems into chunks and calculate mean MAE per group
    mae_by_problem['group'] = (mae_by_problem['problem'] // group_size_problems).astype(int)
    grouped_mae = mae_by_problem.groupby('group').agg({
        'mae_mean': ['mean', 'std'],
        'count': 'sum'
    }).reset_index()
    
    # Flatten column names
    grouped_mae.columns = ['group', 'mean', 'std', 'total_transitions']
    
    # Create group labels
    grouped_mae['label'] = grouped_mae['group'].apply(
        lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
    )
    
    print(f"Model Training Progress - Reward Model MAE (group size = {group_size_problems} problems):")
    print(grouped_mae[['label', 'mean', 'std', 'total_transitions']])
    
    # Create bar chart with error bars
    fig = go.Figure(data=[
        go.Bar(
            x=grouped_mae['label'],
            y=grouped_mae['mean'],
            error_y=dict(
                type='data',
                array=grouped_mae['std'],
                visible=True
            ),
            marker=dict(
                color='rgba(244, 67, 54, 0.8)',
                line=dict(color='rgba(200, 30, 20, 1)', width=2)
            ),
            text=[f"{v:.4f}" for v in grouped_mae['mean']],
            textposition='outside',
            hovertemplate='<b>%{x}</b><br>Mean MAE: %{y:.5f}<br>Std Dev: %{error_y.array:.5f}<br>Transitions: %{customdata}<extra></extra>',
            customdata=grouped_mae['total_transitions']
        )
    ])
    
    fig.update_layout(
        title=f"Model Training Progress: Reward Model MAE Across 100 Conversations (Grouped by {group_size_problems} Problems)",
        xaxis_title="Training Conversation Progress",
        yaxis_title="Mean Absolute Error (Actual vs Predicted Reward)",
        height=500,
        width=1000,
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        font=dict(size=12),
        showlegend=False
    )
    
    fig.show()
    
    # Print summary statistics
    print(f"\n{'='*60}")
    print(f"Training Progress Summary - MAE (group size = {group_size_problems} problems):")
    print(f"{'='*60}")
    print(f"Total groups: {len(grouped_mae)}")
    print(f"First group (early training) mean MAE: {grouped_mae.iloc[0]['mean']:.5f}")
    print(f"Last group (late training) mean MAE: {grouped_mae.iloc[-1]['mean']:.5f}")
    improvement = grouped_mae.iloc[0]['mean'] - grouped_mae.iloc[-1]['mean']
    print(f"MAE reduction (improvement): {improvement:+.5f}")
    best_idx = grouped_mae['mean'].idxmin()
    print(f"Best group (lowest MAE): {grouped_mae.loc[best_idx, 'label']} ({grouped_mae.loc[best_idx, 'mean']:.5f})")
    worst_idx = grouped_mae['mean'].idxmax()
    print(f"Worst group (highest MAE): {grouped_mae.loc[worst_idx, 'label']} ({grouped_mae.loc[worst_idx, 'mean']:.5f})")
else:
    print("No rows with predicted rewards found")

# DIAGNOSTICS: Reward Signal & Learning Quality Analysis

1. **Reward Signal Strength** - Is the reward varied enough to learn from?
2. **Action Imbalance** - Do different actions receive different rewards?
3. **Reward-Success Correlation** - Does our reward signal actually capture student learning?
4. **Early vs Late Training** - Does the model improve over time?
5. **State Distribution** - Is the state space diverse enough for learning?

**What to look for**: Red flags include uniform reward values, actions with identical mean rewards, weak correlations, or state clustering.

In [ ]:
print("="*70)
print("DIAGNOSIS 1: REWARD SIGNAL STRENGTH")
print("="*70)
print("\nWhat to look for:")
print("  • Reward should span a wide range (e.g., -1 to +1, not all 0.15±0.02)")
print("  • High std dev indicates the model has signal to learn from")
print("  • If all rewards cluster near 0, the agent has no incentive to choose actions differently")
print("\nReward Statistics:")
print(f"  Count: {len(df_memory)}")
print(f"  Mean: {df_memory['reward'].mean():.4f}")
print(f"  Std Dev: {df_memory['reward'].std():.4f}")
print(f"  Min: {df_memory['reward'].min():.4f}")
print(f"  Max: {df_memory['reward'].max():.4f}")
print(f"  Median: {df_memory['reward'].median():.4f}")
print(f"  25th percentile: {df_memory['reward'].quantile(0.25):.4f}")
print(f"  75th percentile: {df_memory['reward'].quantile(0.75):.4f}")

# Histogram of rewards
fig = go.Figure(data=[
    go.Histogram(
        x=df_memory['reward'],
        nbinsx=30,
        marker=dict(color='rgba(100, 150, 255, 0.7)', line=dict(color='darkblue', width=1)),
        name='Reward Distribution'
    )
])
fig.update_layout(
    title="Reward Signal Distribution (All Transitions)",
    xaxis_title="Reward Value",
    yaxis_title="Frequency",
    height=400,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=11)
)
fig.show()

print(f"\n✓ If histogram is narrow/single peak → WEAK SIGNAL (problem)")
print(f"✓ If histogram is wide/bimodal → GOOD SIGNAL (desired)")

In [ ]:
print("\n" + "="*70)
print("DIAGNOSIS 2: ACTION BALANCE & MEAN REWARDS BY ACTION")
print("="*70)
print("\nWhat to look for:")
print("  • Actions should be balanced (don't want 80% one action, 10% each other)")
print("  • Mean rewards should DIFFER between actions (e.g., A=0.2, B=0.1, C=0.05)")
print("  • If all actions have nearly identical mean rewards → model can't learn to prefer one")
print("  • Large std dev within action → high variance, harder to learn\n")

ACTIONS = ["SOCRATIC_PROBE", "CONCEPTUAL_HINT", "STRUCTURAL_SCAFFOLD"]
# Action distribution and mean rewards
action_stats = df_memory.groupby('action').agg({
    'reward': ['count', 'mean', 'std', 'min', 'max'],
    'student_success': 'mean',
    'pedagogical_quality': 'mean'
}).round(4)
print("Action Statistics:")
print(action_stats)

# Calculate percentages
action_counts_total = df_memory['action'].value_counts()
print("\nAction Counts & Percentages:")
for action in ACTIONS:
    count = action_counts_total.get(action, 0)
    pct = (count / len(df_memory)) * 100
    mean_reward = df_memory[df_memory['action'] == action]['reward'].mean()
    print(f"  {action:20s}: {count:4d} ({pct:5.1f}%)  |  mean_reward={mean_reward:7.4f}")

# Visualize action rewards with error bars
action_reward_stats = df_memory.groupby('action')['reward'].agg(['mean', 'std', 'count']).reindex(ACTIONS)

fig = go.Figure(data=[
    go.Bar(
        x=action_reward_stats.index,
        y=action_reward_stats['mean'],
        error_y=dict(
            type='data',
            array=action_reward_stats['std'],
            visible=True
        ),
        marker=dict(
            color=['rgba(100, 149, 237, 0.8)', 'rgba(144, 238, 144, 0.8)', 'rgba(255, 165, 0, 0.8)'],
            line=dict(color='black', width=1.5)
        ),
        text=[f"n={int(c)}" for c in action_reward_stats['count']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Mean Reward: %{y:.4f}<br>Std: %{error_y.array:.4f}<extra></extra>'
    )
])

fig.update_layout(
    title="Mean Reward by Pedagogical Action (with ±1 Std Dev)",
    xaxis_title="Pedagogical Action",
    yaxis_title="Mean Reward",
    height=400,
    width=800,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    showlegend=False
)
fig.show()

print(f"\n✓ If all means are similar (within 0.01) → WEAK SIGNAL (problem)")
print(f"✓ If means differ by >0.05 → GOOD SIGNAL (desired)")
print(f"✓ If one action >70% of data → IMBALANCE (exploration too low?)")

In [ ]:
print("\n" + "="*70)
print("DIAGNOSIS 3: REWARD-SUCCESS CORRELATION")
print("="*70)
print("\nWhat to look for:")
print("  • Reward SHOULD correlate with student_success (higher code success = higher reward)")
print("  • Correlation coefficient should be >0.3 (moderate) or >0.5 (strong)")
print("  • If correlation is 0 or negative → reward signal is not capturing learning\n")

# Compute correlations
corr_reward_success = df_memory['reward'].corr(df_memory['student_success'])
corr_reward_pedagogy = df_memory['reward'].corr(df_memory['pedagogical_quality'])
corr_success_pedagogy = df_memory['student_success'].corr(df_memory['pedagogical_quality'])

print(f"Correlation: reward ↔ student_success: {corr_reward_success:.4f}")
print(f"Correlation: reward ↔ pedagogical_quality: {corr_reward_pedagogy:.4f}")
print(f"Correlation: student_success ↔ pedagogical_quality: {corr_success_pedagogy:.4f}")

# Scatter: student_success vs reward
fig = go.Figure(data=[
    go.Scatter(
        x=df_memory['student_success'],
        y=df_memory['reward'],
        mode='markers',
        marker=dict(
            size=6,
            color=df_memory['i'],  # Color by turn index
            colorscale='Viridis',
            colorbar=dict(title="Turn (i)"),
            line=dict(color='white', width=0.5),
            opacity=0.7
        ),
        text=[f"Problem {p}, Turn {i}" for p, i in zip(df_memory['problem'], df_memory['i'])],
        hovertemplate='<b>Turn %{text}</b><br>Student Success: %{x:.2%}<br>Reward: %{y:.4f}<extra></extra>'
    )
])

fig.update_layout(
    title=f"Reward vs Student Success (Correlation: {corr_reward_success:.3f})",
    xaxis_title="Student Code Success Rate (% tests passed)",
    yaxis_title="Reward Signal",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=11),
    xaxis=dict(tickformat='.0%')
)
fig.show()

# Scatter: pedagogical_quality vs reward
fig = go.Figure(data=[
    go.Scatter(
        x=df_memory['pedagogical_quality'],
        y=df_memory['reward'],
        mode='markers',
        marker=dict(
            size=6,
            color=df_memory['student_level'],
            colorscale='Plasma',
            colorbar=dict(title="Student Level"),
            line=dict(color='white', width=0.5),
            opacity=0.7
        ),
        text=[f"Problem {p}, Turn {i}" for p, i in zip(df_memory['problem'], df_memory['i'])],
        hovertemplate='<b>Turn %{text}</b><br>Pedagogical Quality: %{x:.2f}<br>Reward: %{y:.4f}<extra></extra>'
    )
])

fig.update_layout(
    title=f"Reward vs Pedagogical Quality (Correlation: {corr_reward_pedagogy:.3f})",
    xaxis_title="Pedagogical Quality Score",
    yaxis_title="Reward Signal",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=11)
)
fig.show()

print(f"\n✓ If correlations <0.1 → Your reward may not capture what you want (problem)")
print(f"✓ If correlations >0.3 → GOOD SIGNAL (desired)")

In [ ]:
print("\n" + "="*70)
print("DIAGNOSIS 4: EARLY VS LATE TRAINING COMPARISON")
print("="*70)
print("\nWhat to look for:")
print("  • Early problems (0-25): Baseline performance (model learning from scratch)")
print("  • Late problems (75-99): Should show improvement IF learning is working")
print("  • Compare: mean reward, accuracy, action distribution, model MAE")
print("  • If early ≈ late → Model NOT learning from experience\n")

# Split into early, mid, late
early_idx = df_memory['problem'] < 25
mid_idx = (df_memory['problem'] >= 25) & (df_memory['problem'] < 75)
late_idx = df_memory['problem'] >= 75

early_data = df_memory[early_idx]
mid_data = df_memory[mid_idx]
late_data = df_memory[late_idx]

print("EARLY PROBLEMS (0-24):")
print(f"  Transitions: {len(early_data)}")
print(f"  Mean reward: {early_data['reward'].mean():.4f}")
print(f"  Mean accuracy: {early_data['student_success'].mean():.2%}")
print(f"  Mean pedagogical quality: {early_data['pedagogical_quality'].mean():.4f}")

print("\nMID PROBLEMS (25-74):")
print(f"  Transitions: {len(mid_data)}")
print(f"  Mean reward: {mid_data['reward'].mean():.4f}")
print(f"  Mean accuracy: {mid_data['student_success'].mean():.2%}")
print(f"  Mean pedagogical quality: {mid_data['pedagogical_quality'].mean():.4f}")

print("\nLATE PROBLEMS (75-99):")
print(f"  Transitions: {len(late_data)}")
print(f"  Mean reward: {late_data['reward'].mean():.4f}")
print(f"  Mean accuracy: {late_data['student_success'].mean():.2%}")
print(f"  Mean pedagogical quality: {late_data['pedagogical_quality'].mean():.4f}")

# Calculate improvements
reward_improvement = late_data['reward'].mean() - early_data['reward'].mean()
accuracy_improvement = late_data['student_success'].mean() - early_data['student_success'].mean()

print(f"\nCHANGE (Late - Early):")
print(f"  Reward: {reward_improvement:+.4f}")
print(f"  Accuracy: {accuracy_improvement:+.2%}")

# Visualization: Rolling mean of key metrics
df_sorted = df_memory.sort_values('problem').reset_index(drop=True)
window = max(10, len(df_sorted) // 20)  # Adaptive window size

rolling_reward = df_sorted['reward'].rolling(window=window, center=True).mean()
rolling_accuracy = df_sorted['student_success'].rolling(window=window, center=True).mean()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Mean Reward Over Training", "Student Accuracy Over Training"),
    vertical_spacing=0.12
)

fig.add_trace(
    go.Scatter(
        x=df_sorted['problem'],
        y=rolling_reward,
        mode='lines',
        name='Reward (rolling avg)',
        line=dict(color='orange', width=3),
        hovertemplate='Problem %{x}: Reward = %{y:.4f}<extra></extra>'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_sorted['problem'],
        y=rolling_accuracy,
        mode='lines',
        name='Accuracy (rolling avg)',
        line=dict(color='green', width=3),
        hovertemplate='Problem %{x}: Accuracy = %{y:.2%}<extra></extra>'
    ),
    row=2, col=1
)

fig.update_xaxes(title_text="Problem Index (0-99)", row=2, col=1)
fig.update_yaxes(title_text="Mean Reward", row=1, col=1)
fig.update_yaxes(title_text="Mean Accuracy", row=2, col=1)

fig.update_layout(
    title_text=f"Training Progress: Rolling Average (window={window})",
    height=600,
    width=1000,
    hovermode='x unified',
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=11)
)
fig.show()

print(f"\n✓ If both lines are flat/noisy → Model NOT learning (problem)")
print(f"✓ If late > early clearly → LEARNING DETECTED (desired)")

In [ ]:
print("\n" + "="*70)
print("DIAGNOSIS 5: STATE SPACE DISTRIBUTION")
print("="*70)
print("\nWhat to look for:")
print("  • Turn index (i): Should be distributed 0-9 (more 0-2 normal due to early stopping)")
print("  • Student/Tutor levels: Should span 1-4 to cover diverse scenarios")
print("  • If all states cluster in one corner → Model can't learn diverse behaviors\n")

# Turn distribution
print("TURN INDEX DISTRIBUTION (i):")
turn_counts = df_memory['i'].value_counts().sort_index()
for turn, count in turn_counts.items():
    pct = (count / len(df_memory)) * 100
    bar = '█' * int(pct / 2)
    print(f"  Turn {turn}: {count:4d} ({pct:5.1f}%) {bar}")

# Level distributions
print("\nSTUDENT LEVEL DISTRIBUTION (1-4 scale):")
level_counts = df_memory['last_student_level'].value_counts().sort_index()
for level, count in level_counts.items():
    pct = (count / len(df_memory)) * 100
    bar = '█' * int(pct / 2)
    print(f"  Level {level}: {count:4d} ({pct:5.1f}%) {bar}")

print("\nTUTOR LEVEL DISTRIBUTION (1-4 scale):")
level_counts = df_memory['last_tutor_level'].value_counts().sort_index()
for level, count in level_counts.items():
    pct = (count / len(df_memory)) * 100
    bar = '█' * int(pct / 2)
    print(f"  Level {level}: {count:4d} ({pct:5.1f}%) {bar}")

# Visualizations
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Turn Index Distribution", "Student Level Distribution",
                    "Tutor Level Distribution", "(Student Level, Tutor Level) Heatmap"),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "heatmap"}]]
)

# Turn distribution
turn_data = df_memory['i'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=turn_data.index, y=turn_data.values, marker=dict(color='rgba(100, 150, 255, 0.7)'),
           name='Turn Count', hovertemplate='Turn %{x}: %{y}<extra></extra>'),
    row=1, col=1
)

# Student level distribution
student_level_data = df_memory['last_student_level'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=student_level_data.index, y=student_level_data.values, marker=dict(color='rgba(144, 238, 144, 0.7)'),
           name='Student Level Count', hovertemplate='Level %{x}: %{y}<extra></extra>'),
    row=1, col=2
)

# Tutor level distribution
tutor_level_data = df_memory['last_tutor_level'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=tutor_level_data.index, y=tutor_level_data.values, marker=dict(color='rgba(255, 165, 0, 0.7)'),
           name='Tutor Level Count', hovertemplate='Level %{x}: %{y}<extra></extra>'),
    row=2, col=1
)

# Cross-tabulation heatmap
state_heatmap = pd.crosstab(df_memory['last_student_level'], df_memory['last_tutor_level'])
fig.add_trace(
    go.Heatmap(z=state_heatmap.values, x=state_heatmap.columns, y=state_heatmap.index,
               colorscale='Blues', name='Count',
               hovertemplate='Student %{y} → Tutor %{x}: %{z}<extra></extra>'),
    row=2, col=2
)

fig.update_xaxes(title_text="Turn Index (i)", row=1, col=1)
fig.update_xaxes(title_text="Student Level", row=1, col=2)
fig.update_xaxes(title_text="Tutor Level", row=2, col=1)
fig.update_xaxes(title_text="Tutor Level", row=2, col=2)

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_yaxes(title_text="Count", row=2, col=1)
fig.update_yaxes(title_text="Student Level", row=2, col=2)

fig.update_layout(
    title_text="State Space Distribution Analysis",
    height=700,
    width=1200,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=10),
    showlegend=False
)
fig.show()

print(f"\n✓ If all distributions cluster in one region → STATE COLLAPSE (problem)")
print(f"✓ If distributions span 1-4 with variety → DIVERSE STATE SPACE (desired)")
print(f"✓ More turns 0-2 than 3-9 is EXPECTED (early stopping is normal)")

In [ ]:
print("\n" + "="*70)
print("DIAGNOSIS 6: REWARD MODEL PREDICTION QUALITY")
print("="*70)
print("\nWhat to look for:")
print("  • Model MAE: Lower is better (model predicting rewards accurately)")
print("  • If MAE >> reward std → Predictions are poor")
print("  • If MAE << reward std → GOOD PREDICTIONS (desired)")
print("  • Check if predictions improve over time\n")

# Compute MAE for predictions
pred_columns_list = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
df_with_preds = df_memory[pred_columns_list + ['reward']].dropna()

if len(df_with_preds) > 0:
    # For each row, use the prediction of the action that was actually taken
    def get_predicted_reward_for_action(row):
        action = df_memory.loc[row.name, 'action']
        pred_col = f'pred_reward_{action}'
        if pred_col in df_with_preds.columns:
            return df_with_preds.loc[row.name, pred_col]
        return None
    
    # Simpler approach: use max prediction for each row
    best_pred = df_with_preds[pred_columns_list].max(axis=1)
    mae = (best_pred - df_with_preds['reward']).abs().mean()
    
    reward_std = df_memory['reward'].std()
    ratio = mae / reward_std if reward_std > 0 else float('inf')
    
    print(f"Rows with predictions: {len(df_with_preds)} / {len(df_memory)}")
    print(f"Mean Absolute Error (MAE): {mae:.6f}")
    print(f"Reward Std Dev: {reward_std:.6f}")
    print(f"MAE / Reward Std Ratio: {ratio:.4f}")
    
    if ratio > 1.0:
        print(f"⚠ HIGH RATIO: Model predictions are worse than random guessing!")
    elif ratio > 0.5:
        print(f"⚠ MODERATE RATIO: Model predictions have room for improvement")
    else:
        print(f"✓ GOOD RATIO: Model is making reasonable predictions")
    
    # Plot MAE over training time
    df_with_mae = df_with_preds.copy()
    df_with_mae['problem'] = df_memory.loc[df_with_mae.index, 'problem'].values
    df_with_mae['mae'] = (best_pred - df_with_preds['reward']).abs()
    
    mae_by_problem = df_with_mae.groupby('problem')['mae'].mean()
    
    fig = go.Figure(data=[
        go.Scatter(
            x=mae_by_problem.index,
            y=mae_by_problem.values,
            mode='lines+markers',
            name='MAE',
            line=dict(color='red', width=2),
            marker=dict(size=6),
            hovertemplate='Problem %{x}: MAE = %{y:.6f}<extra></extra>'
        )
    ])
    
    fig.add_hline(y=mae, line_dash="dash", line_color="red", annotation_text=f"Mean MAE: {mae:.6f}")
    
    fig.update_layout(
        title="Reward Model MAE Over Training",
        xaxis_title="Problem Index (0-99)",
        yaxis_title="Mean Absolute Error",
        height=400,
        width=1000,
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        font=dict(size=11)
    )
    fig.show()
    
    print(f"\n✓ If MAE decreases over time → Model is improving")
    print(f"✓ If MAE is flat/high → Model not learning or insufficient data")
else:
    print("No predictions found in memory (cold start phase?)")

## Diagnostic Summary & Next Steps

Based on the diagnostics above, here are the most likely issues and potential fixes:

### Issue 1: Weak Reward Signal
**Symptom**: All rewards cluster near 0.15-0.25, with low variance
- **Root cause**: Reward components (student_success, pedagogical_quality, code_runs) are all contributing small amounts
- **Fixes**:
  - Increase weighting of student_success (increase 1-LAMBDA from 0.3)
  - Add bonuses for reaching later turns (agent persisting helps)
  - Consider binary reward (solved vs not solved) instead of continuous

### Issue 2: Actions Have Identical Rewards
**Symptom**: SOCRATIC_PROBE ≈ CONCEPTUAL_HINT ≈ STRUCTURAL_SCAFFOLD (all ≈0.18)
- **Root cause**: Different pedagogical moves may not actually affect student outcomes in simulation
- **Fixes**:
  - Verify judge agents are correctly evaluating tutor behavior
  - Add explicit reward bonus for actions that match student level
  - Increase sensitivity of pedagogy quality metric to action type

### Issue 3: Poor Correlation (Reward ↔ Student Success)
**Symptom**: Scatter plot is random cloud, correlation < 0.1
- **Root cause**: Reward formula doesn't capture learning
- **Fixes**:
  - Simplify reward: `reward = student_success + leakage_penalty`
  - Focus on cumulative progress, not turn-by-turn success
  - Add larger penalties for bad outcomes (leakage, changing problem)

### Issue 4: No Learning Curve (Early ≈ Late)
**Symptom**: Mean reward flat across problems 0-99
- **Root cause**: Insufficient training data or model can't learn from state features
- **Fixes**:
  - Increase retrain frequency vs data volume
  - Add interaction features (e.g., student_level × tutor_level)
  - Add more context to state (e.g., problem difficulty from LeetCode)
  - Lower epsilon exploration to let learned policy take effect

### Issue 5: State Space Collapse
**Symptom**: All states cluster at (student_level=1, tutor_level=1, i=0-2)
- **Root cause**: Early stopping dominates, or judges not calibrated
- **Fixes**:
  - Adjust stop conditions (let conversations run longer)
  - Recalibrate judge prompts for more varied level assignments
  - Add problem difficulty to state representation

### Issue 6: Model MAE Too High
**Symptom**: MAE >> reward_std (predictions worse than baseline)
- **Root cause**: Insufficient data or RF overfitting
- **Fixes**:
  - Use simpler model (linear regression instead of RF)
  - Add regularization to RF
  - Collect more data (train longer)
  - Feature engineering: normalize state features

In [ ]:
## Feature importance

## Reward Model Feature Importance Analysis

In [ ]:
import pickle

print("="*70)
print("REWARD MODEL ANALYSIS: FEATURE IMPORTANCE")
print("="*70)
print("\nWhat this shows:")
print("  • Which state features have the most influence on predicted rewards")
print("  • Higher importance = feature is more useful for decision-making")
print("  • If all importances are equal → model treats all features as useless\n")

# Load the trained reward model
model_path = RUN_PATH / "checkpoints" / "reward_model.pkl"

if model_path.exists():
    with open(model_path, 'rb') as f:
        reward_model = pickle.load(f)
    
    print(f"✓ Loaded reward model from {model_path}")
    
    # The model is a Pipeline with preprocessing and Random Forest
    # Get the Random Forest estimator
    rf_model = reward_model.named_steps['reg']
    
    # Get feature importances
    importances = rf_model.feature_importances_
    
    # Get feature names from the preprocessor
    preprocessor = reward_model.named_steps['pre']
    
    # After OneHotEncoder on categorical features, we need to get the feature names
    # Categorical features: last_action (3 values), action (3 values) = 6 one-hot features
    # Numerical features: i, last_student_level, last_tutor_level, last_reward = 4 features
    
    categorical_names = []
    for i, cat in enumerate(preprocessor.named_transformers_['cat'].get_feature_names_out()):
        categorical_names.append(f"cat_{cat}")
    
    # numerical_names = ['i', 'last_student_level', 'last_tutor_level', 'last_reward']
    numerical_names =  ["i", "last_student_level", "last_tutor_level", "last_reward", "last_coding_score"]
    
    all_feature_names = categorical_names + numerical_names
    
    print(f"\nTotal features in model: {len(all_feature_names)}")
    print(f"  Categorical (one-hot encoded): {len(categorical_names)}")
    print(f"  Numerical: {len(numerical_names)}")
    
    # Create DataFrame for easier analysis
    feature_importance_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print(f"\nFeature Importances (sorted by importance):")
    print(feature_importance_df.to_string(index=False))
    
    # Sum importances by feature type
    print(f"\nImportance by Type:")
    cat_importance = feature_importance_df[feature_importance_df['Feature'].str.startswith('cat_')]['Importance'].sum()
    num_importance = feature_importance_df[feature_importance_df['Feature'].str.startswith(('i', 'last_'))]['Importance'].sum()
    print(f"  Categorical features: {cat_importance:.4f}")
    print(f"  Numerical features: {num_importance:.4f}")
    
    # Visualization: Bar chart of feature importances
    fig = go.Figure(data=[
        go.Bar(
            x=feature_importance_df['Feature'],
            y=feature_importance_df['Importance'],
            marker=dict(
                color=feature_importance_df['Importance'],
                colorscale='Viridis',
                line=dict(color='black', width=1)
            ),
            text=[f"{v:.4f}" for v in feature_importance_df['Importance']],
            textposition='outside',
            hovertemplate='<b>%{x}</b><br>Importance: %{y:.6f}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title="Reward Model Feature Importances (Random Forest)",
        xaxis_title="Feature",
        yaxis_title="Importance Score",
        height=500,
        width=1000,
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        font=dict(size=11),
        xaxis=dict(tickangle=-45),
        showlegend=False
    )
    
    fig.show()
    
    # Interpretation
    top_feature = feature_importance_df.iloc[0]
    print(f"\n✓ Top feature: {top_feature['Feature']} (importance: {top_feature['Importance']:.4f})")
    print(f"✓ Model uses {'categorical' if cat_importance > num_importance else 'numerical'} features more")
    
    if feature_importance_df['Importance'].max() < 0.15:
        print("⚠ WARNING: All features have low importance → Model may not be learning meaningful patterns")
else:
    print(f"✗ Model file not found at {model_path}")
    print("  Make sure the training has completed and model was saved")